In [ ]:
#installing kaggle library
! pip install kaggle

In [ ]:
# configuring path for kaggle.json file

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle
!chmod 600 ~/.kaggle/kaggle.json

**Importing dataset using api**

In [ ]:
# API to fetch dataset from kaggle
!kaggle datasets download -d kazanova/sentiment140

In [ ]:
# extracting compressed dataset

from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset,'r') as zip :
  zip.extractall()
  print('The dataset is extracted ')



# **Importing the dependencies**

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
# downloading the stopwords
import nltk
nltk.download('stopwords')

In [ ]:
# printing the stopwords
# those words which doent have more contextual importance for the analysis
print(stopwords.words('english'))

# **Data Processing**


In [ ]:
# loading data into pandas from csv file for the analysis purpose
t_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',encoding='ISO-8859-1')

In [ ]:
# checking the dimensions of data
t_data.shape


In [ ]:
# printing first 5 rows of data
t_data.head()


In [ ]:
column_names = ['target','id','date','flag','user','text']
t_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',names = column_names,encoding='ISO-8859-1')

In [ ]:
# printing first 5 rows of data
t_data.head()

In [ ]:
# printing last 5 rows of data
t_data.tail()

In [ ]:
# checking for null/missing values
t_data.isnull().sum()


In [ ]:
# checking the distributin of the target variable
t_data['target'].value_counts()

In [ ]:
# converting the target label from 4 to 1

t_data.replace({'target':{4:1}},inplace = True)

In [ ]:
# checking the distributin of the target variable
t_data['target'].value_counts()

# **Stemming**
it is a process of reducing a word to its root word

example : actor,actress,acting = act

act is the root word

In [ ]:
port_stem = PorterStemmer()


In [ ]:
pattern =re.compile('[^a-zA-Z]')

In [ ]:
english_stopwords = stopwords.words('english')

In [ ]:
port_stemmer = PorterStemmer()



In [ ]:
def stemming(content):
      stemmed_content = re.sub(pattern,' ',content)
      stemmed_content = stemmed_content.lower()
      stemmed_content = stemmed_content.split()
      stemmed_content = [port_stemmer.stem(word) for word in stemmed_content if not word in english_stopwords]
      stemmed_content = ' '.join(stemmed_content)

      return stemmed_content

In [ ]:
t_data['stemmed_content'] = t_data['text'].apply(stemming)

In [ ]:
t_data.head()

In [ ]:
print(t_data['stemmed_content'])


In [ ]:
# seperarting data and values

x = t_data['stemmed_content'].values
y = t_data['target'].values

In [ ]:
print(x)

In [ ]:
print(y)

# **Splitting data into training and testing**

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(x,y,test_size = 0.2,stratify = y,random_state = 2)

In [ ]:
print(x.shape,X_train.shape,X_test.shape)

In [ ]:
print(X_train)

In [ ]:
print(X_test)

In [ ]:
# converting the textual data to numerical data
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
print(X_train)

In [ ]:
print(X_test)

# **Training The Model**
**Logistic Regression Model**

In [ ]:
model = LogisticRegression(max_iter = 1000)

In [ ]:
model.fit(X_train,Y_train)

# **Model Evaluation**


In [ ]:
# accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction,Y_train)

In [ ]:
print('accuracy score on training data : ', training_data_accuracy)

In [ ]:
# accuracy score on the testing data
X_test_prediction = model.predict(X_test)
testing_data_accuracy = accuracy_score(X_test_prediction,Y_test)

In [ ]:
print('accuracy score on testing data : ', testing_data_accuracy)

**Model Accuracy : 77.80 %**

# **Saving the trained Model :**

In [ ]:
import pickle

In [ ]:
filename = 'trained_model.pkl'
pickle.dump(model,open(filename,'wb'))


using the saved model for future predictions


In [ ]:
# loading the saved model
loaded_model = pickle.load(open('/content/trained_model.pkl','rb'))

In [ ]:
X_new = X_test[3]

prediction = loaded_model.predict(X_new)
print(prediction)

if (prediction[0]==0):
  print('Negative Tweet')
else:
  print('Positive Tweet')

print('for checking if the prediction is right or wrong :: ',Y_test[3])